# Phase 5: test evaluation and relearning attacks (Gate 5)

**Only after Gate 4 is approved.** Upload the latest `colab/code_bundle.tar.gz` first.
The test splits are read here for the first time, once per model (a second access is refused).
Relearning: θ_ref and the selected M1/M2/M3 models are fine-tuned for 300 steps, with Portuguese only (n0)
and with 1M characters of fresh English (fresh1M).

In [ ]:
# 1. Settings + GPU check
SEED = 0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive. MyDrive/unlearn/{checkpoints,results} is the working copy from Phase 3 on.
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/unlearn'
import os
for d in ('checkpoints', 'results', 'gate_packs'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print(sorted(os.listdir(DRIVE)))

In [ ]:
# 3. Data bundle (unchanged since Phase 0) + latest code bundle on top; check the data SHAs;
#    link checkpoints/ and results/ to Drive so every script writes there directly.
!rm -rf /content/unlearn && mkdir -p /content/unlearn
!tar xzf {DRIVE}/unlearn_bundle.tar.gz -C /content/unlearn
!tar xzf {DRIVE}/code_bundle.tar.gz -C /content/unlearn
%cd /content/unlearn
!cd data/processed && sha256sum -c --quiet MANIFEST.sha256 && echo "data OK"
!rm -rf checkpoints results && ln -s {DRIVE}/checkpoints checkpoints && ln -s {DRIVE}/results results

In [ ]:
# 4. Dependencies + tests. tests/test_data.py is not run here: it needs the *_titles.txt files, which are
#    not in the bundle. It passed on the Mac at Gate 0, and cell 3 already proved the .bin files are
#    byte-identical to the Mac copy (SHA-256). Smoke-data tests are skipped (no smoke data on Colab).
!pip -q install pyyaml
!python -m pytest -q --ignore=tests/test_data.py 2>&1 | tail -2

In [ ]:
# 5. Bring the Mac's results into Drive (never overwrites), then re-hash every protected checkpoint.
#    Drive cannot chmod, so this SHA check is what protects θ_ref. It must print OK, never raise.
!python scripts/colab_sync.py merge --src results_mac --dst results
!python scripts/colab_sync.py verify

In [ ]:
# 6. Test-split evaluation of every model: bpc + 95% CI, KL to θ_ref, leakage from test prompts
!python scripts/phase5_eval.py --seed {SEED} 2>&1 | grep -v Warning

In [ ]:
# 7. Relearning attacks (finished runs are skipped)
!python scripts/relearn.py --seed {SEED} --with-oracle 2>&1 | grep -v Warning

In [ ]:
# 8. Results table + relearning figure. Paste the printed table into the chat for Gate 5.
import json
pts = json.load(open(f'results/operating_points_s{SEED}.json'))
names = [f'theta_ref_s{SEED}@test', f'theta_b_s{SEED}@test', f'theta_en_s{SEED}@test']
for run, info in pts['runs'].items():
    names.append(f'{run}@prac')
    if run in pts['selected'].values() and info['oracle']:
        names.append(f'{run}@oracle')
NAMES = ' '.join(names)
!python scripts/plots.py table --names {NAMES}
!python scripts/plots.py relearn --seed {SEED}
from IPython.display import Image; Image(f'results/figures/relearn_s{SEED}.png')

In [ ]:
# Gate pack: re-check the hashes, then zip results/ (small files only, no checkpoints).
# Download MyDrive/unlearn/gate_packs/gate5_results.zip and unzip it into the project as colab_results/gate5/
!python scripts/colab_sync.py verify
!cd {DRIVE} && rm -f gate_packs/gate5_results.zip && zip -qr gate_packs/gate5_results.zip results && ls -la gate_packs